# README

- **Author**: RUEI-HSIANG, CHANG（B12303030；使用 AI 協助資料準備）
- **Created At**: 2026-10-17
- **Last Modified At**: 2026-10-17

---
## What does this file do?
將官方登記來源表轉為一人一列的候選人工作名冊；標準化民國日期與「無」推薦政黨標記，建立可追溯 ID，與 2001–2022 歷史縣市長資料產生姓名核對清單。保留全部 81 位登記者，不套用尚未確認的樣本排除規則，不做模型或 train/test 切分。

兩種首次參選：`first_time_mayor_lifetime` 與 `first_time_any_elected_office`，皆採 1=首次、0=非首次、空白=未知。姓名相同不代表身分核實；沒有歷史命中不代表首次。兩欄本次保持未知，待逐人來源核對。歷史性別／出生年／學歷只放在 evidence，不自動回填 2026。`age_2026_minus_birth_year` 沿用提案的年度年齡定義，非投票日足歲。官方社群未蒐集不能填 0。

---
## What does this file take?
四份固定輸入（下列程式逐檔綁定 SHA-256），文件為 [sources_2026_documentation.ipynb](../raw/sources_2026_documentation.ipynb)。兩 PDF 留存原文、CSV 由 extract_registration_pdfs.py 轉檔；歷史表為原包固定快照。

---
## What does this file output?
三份 `data/processed/` CSV：candidates_2026_registration（81 人）、candidate_prior_mayor_name_matches（姓名比對線索，非認定）、campaign_collection_worklist（81 人的待蒐集工作表）。Role 為 preparation，不是已完成分析輸入。各檔文件：[candidates_2026_documentation.ipynb](../processed/candidates_2026_documentation.ipynb)。下方執行後重讀檢查筆數、欄位、內容及鍵並列印 hash；詳細欄位定義見同份文件。允許已授權的同名輸出重建，原始輸入不變。


In [1]:
from pathlib import Path
import hashlib
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "AGENTS.md").is_file() and (candidate / "templates").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook within the student project containing AGENTS.md and templates/.")


PROJECT_ROOT = find_project_root()


def project_file(relative_path, area, must_exist=True):
    # Explicit project-relative paths only; never select the first matching file.
    rel = Path(relative_path)
    if rel.is_absolute() or ".." in rel.parts or "<" in str(rel):
        raise ValueError(f"Replace the placeholder with an exact project-relative path: {relative_path}")
    path = (PROJECT_ROOT / rel).resolve()
    allowed = (PROJECT_ROOT / "data" / area).resolve()
    if allowed == PROJECT_ROOT or not allowed.is_relative_to(PROJECT_ROOT):
        raise ValueError(f"data/{area}/ resolves outside the project root.")
    if not path.is_relative_to(allowed):
        raise ValueError(f"Expected a file inside data/{area}/: {relative_path}")
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def verify_input_sha256(path, expected):
    # Bind this run to the exact file inspected when the notebook was prepared.
    if (not isinstance(expected, str) or len(expected) != 64
            or any(char not in "0123456789abcdefABCDEF" for char in expected)):
        raise ValueError("Fill the expected SHA-256 from the inspected input before loading data.")
    actual = sha256_file(path)
    if actual != expected.lower():
        raise ValueError(f"Input changed: identify the new version before loading {path.name}.")
    return actual

EXPECTED = {'2026_metropolitan_registration.pdf': '5ab6ec29145feab84f8e697495f039217bb18ace252382e7e150bb1ea7ae18fa', '2026_county_registration.pdf': '97c32aa206bfd5d1e07d5b9658f853b6b038053b662dd04fdd0132af614a8136', 'cec_registration_2026_source_table.csv': 'c057759cabef3f0b6f7a9137d8e85c9afee2b45358c8c31e764ced4cae8c85f0', 'historical_mayor_candidates_2001_2022.csv': 'ed8f63163d5f0c82301671301e658167cdc4eea5fa81aee8d9078c3226abb097'}
for name, fingerprint in EXPECTED.items():
    verify_input_sha256(project_file("data/raw/"+name,"raw"), fingerprint)
print("All four source fingerprints verified")

All four source fingerprints verified


In [2]:
OUTPUTS={'candidates_2026_registration': 'data/processed/candidates_2026_registration.csv', 'candidate_prior_mayor_name_matches': 'data/processed/candidate_prior_mayor_name_matches.csv', 'campaign_collection_worklist': 'data/processed/campaign_collection_worklist.csv'}
output_paths={k:project_file(v,"processed",must_exist=False) for k,v in OUTPUTS.items()}
assert len(set(output_paths.values()))==3
assert not (set(output_paths.values()) & {project_file("data/raw/"+n,"raw") for n in EXPECTED})

In [3]:
import re
source = pd.read_csv(project_file("data/raw/cec_registration_2026_source_table.csv","raw"),dtype=str,keep_default_na=False)
history = pd.read_csv(project_file("data/raw/historical_mayor_candidates_2001_2022.csv","raw"),dtype=str,keep_default_na=False)
assert len(source)==81 and source.county_city.nunique()==22
assert not source.duplicated(["county_city","candidate_name"]).any()
def norm(name):
    return re.sub(r"\s+", "", name).replace("台", "臺")
history["match_name"] = history.candidate_name.map(norm)
roster = source.copy()
roster.insert(0,"candidate_key",[f"2026-{o}-{i:03d}" for i,o in enumerate(source.office_type,1)])
def roc_date(value):
    y,m,d=map(int,value.split("/")); return f"{y+1911:04d}-{m:02d}-{d:02d}"
roster["registration_date"] = roster.registration_date_roc.map(roc_date)
roster["party_at_registration"] = roster.party_recommendation_original.replace({"無":"無黨籍及未經政黨推薦"})
roster["metropolitan_2026"] = (roster.office_type=="C1").astype(int).astype(str)
roster["qualification_status"] = "pending_official_review_crosscheck"
roster["sample_inclusion"] = "pending_group_rule"
# No inference of 2026 education, incumbency, or identity from a name match.
for field in ["sex_2026","birth_date_verified","age_2026_minus_birth_year","education_2026","incumbency_2026","first_time_mayor_lifetime","first_time_any_elected_office","history_identity_verified"]:
    roster[field]=""
roster["first_time_review_status"]="pending_identity_and_lifetime_history"
evidence=[]
for _, candidate in roster.iterrows():
    hits=history.loc[history.match_name==norm(candidate.candidate_name)].sort_values("election_date")
    idx=roster.index[roster.candidate_key==candidate.candidate_key][0]
    roster.loc[idx,"prior_mayor_name_match_count_2001_2022"]=str(len(hits))
    roster.loc[idx,"prior_mayor_name_match_years"]=";".join(sorted(set(hits.election_year)))
    for _,h in hits.iterrows():
        evidence.append({"candidate_key":candidate.candidate_key,"candidate_name_2026":candidate.candidate_name,"county_city_2026":candidate.county_city,"historical_name":h.candidate_name,"historical_county_city":h.county_city,"historical_election_date":h.election_date,"historical_sex":h.sex,"historical_birth_year":h.birth_year,"historical_education_cec":h.education_level_cec,"historical_education_details":h.education_details,"historical_person_key":h.person_key,"source_url":h.source_url,"source_file_legacy":h.source_file,"match_status":"name_only_pending_identity_review"})
evidence=pd.DataFrame(evidence)
worklist=roster[["candidate_key","county_city","candidate_name"]].copy()
for field in ["official_profile_url","official_facebook_url","official_website_url","platform_verification_evidence","profile_capture_date","sex_source_url","birth_source_url","education_source_url","current_office_source_url","prior_mayor_history_source_url","any_office_history_source_url","bulletin_2026_url","visual_capture_path","follower_count_start","follower_count_end","reviewer","review_notes"]:
    worklist[field]=""
worklist["collection_status"]="not_collected"
outputs={"candidates_2026_registration":roster.fillna(""),"candidate_prior_mayor_name_matches":evidence.fillna(""),"campaign_collection_worklist":worklist}
print({"registrants":len(roster),"counties":roster.county_city.nunique(),"candidates_with_name_matches":int((roster.prior_mayor_name_match_count_2001_2022.astype(int)>0).sum()),"historical_match_rows":len(evidence)})
display(roster[["county_city","candidate_name","party_at_registration","prior_mayor_name_match_count_2001_2022"]].head(10))


{'registrants': 81, 'counties': 22, 'candidates_with_name_matches': 21, 'historical_match_rows': 25}


,county_city,candidate_name,party_at_registration,prior_mayor_name_match_count_2001_2022
0,臺北市,郭璽,台灣麻將最大黨,0
1,臺北市,沈伯洋,民主進步黨,0
2,臺北市,蕭文乾,臺灣SoR無法黨,0
3,臺北市,唐新民,三勢團結促進聯盟,1
4,臺北市,蔣萬安,中國國民黨,1
5,臺北市,林志成,無黨籍及未經政黨推薦,0
6,新北市,蘇巧慧,民主進步黨,0
7,新北市,李四川,中國國民黨,0
8,新北市,蘇輝湟,無黨籍及未經政黨推薦,0
9,桃園市,張善政,中國國民黨,1


In [4]:
verification={}
for key,frame in outputs.items():
    path=output_paths[key]
    if path.exists():
        existing = pd.read_csv(path,dtype=str,keep_default_na=False)
        pd.testing.assert_frame_equal(frame.reset_index(drop=True), existing, check_dtype=False, obj="Existing output changed: preserve manual edits and review before rebuilding")
    frame.to_csv(path,index=False,encoding="utf-8-sig")
    reread=pd.read_csv(path,dtype=str,keep_default_na=False)
    pd.testing.assert_frame_equal(frame.reset_index(drop=True),reread,check_dtype=False)
    assert not reread.duplicated().any()
    if key!="candidate_prior_mayor_name_matches": assert reread.candidate_key.is_unique and len(reread)==81
    verification[key]={"path":OUTPUTS[key],"rows":len(reread),"sha256":sha256_file(path),"status":"read-back passed; incomplete research fields explicitly pending"}
    print(verification[key])
assert (outputs["candidates_2026_registration"][["first_time_mayor_lifetime","first_time_any_elected_office"]]=="").all().all()
for name,fingerprint in EXPECTED.items(): verify_input_sha256(project_file("data/raw/"+name,"raw"),fingerprint)


{'path': 'data/processed/candidates_2026_registration.csv', 'rows': 81, 'sha256': '765797d09de8e70f6cb54486cf660c44f93e7aab1c68c3a6233db60aad7c5832', 'status': 'read-back passed; incomplete research fields explicitly pending'}
{'path': 'data/processed/candidate_prior_mayor_name_matches.csv', 'rows': 25, 'sha256': '8a75f370e91438b5eb1462d328baa60473dbae29964f36fbeb48c8aa8e134c4f', 'status': 'read-back passed; incomplete research fields explicitly pending'}
{'path': 'data/processed/campaign_collection_worklist.csv', 'rows': 81, 'sha256': 'c0986b0282dc2b9911c46f864fff459bef189260db1b0a0fa16c09fe1e5c006d', 'status': 'read-back passed; incomplete research fields explicitly pending'}
